Tutorial 6: Weak Lensing
========================

Our modeling tutorials have concentrated on **strong lensing**: the multiple images, arcs and Einstein rings produced by galaxy, group and cluster lenses.

In the previous tutorial, we used the positions of multiply imaged sources to constrain a cluster’s mass distribution. Those images provide particularly valuable information in the central region where they form. To measure the mass distribution over a much wider region, we can also use **weak lensing**.

The same gravitational field distorts the shapes of background galaxies beyond the region containing prominent arcs. These distortions are usually difficult to distinguish from a galaxy’s unknown intrinsic shape: an elongated galaxy might have been elongated even without lensing.

The signal becomes measurable statistically. Nearby background galaxies experience a coherent lensing distortion, while their intrinsic orientations contribute scatter. Measuring many galaxies allows us to estimate that coherent signal.

This scatter is called **shape noise**. Increasing the number of usable background galaxies reduces its contribution to the uncertainty, although observational biases and correlations between intrinsic galaxy shapes also require attention in real analyses.

You first encountered a weak-lensing dataset in Chapter 1. Here we return to its **shear catalogue**: background-galaxy positions, two components describing the estimated distortion, and their uncertainties.

We simulate a catalogue around a cluster-scale lens, visualize it, inspect a fit using the input mass model, and then infer the model parameters with a non-linear search. The familiar `Tracer`, `Fit`, `Analysis` and search workflow remains central.

This tutorial closes both this chapter and the four core chapters of **HowToLens**.

__Contents__

- **Shear Catalogues:** The measurements used in a weak-lensing analysis.
- **Mass Scales:** Weak lensing around galaxies, groups and clusters.
- **Ray Tracing:** Build a simple cluster-scale mass model.
- **Source Galaxy Positions:** Sample background galaxies across the observed field.
- **Simulate:** Generate a shear catalogue with shape noise.
- **Visualize:** Inspect the shear field and dataset summary.
- **Mass Map:** Reconstruct a convergence map from the catalogue.
- **Fitting:** Inspect the likelihood and residuals for a specified mass model.
- **Model Fit:** Infer mass-model parameters with a non-linear search.
- **Result:** Inspect the inferred model and tangential shear profile.
- **Joint Strong and Weak Lensing:** Combine constraints across different radii.
- **Wrap Up:** Complete the core HowToLens lectures.

__Google Colab Setup__

Set up Google Colab as in [the introductory notebook](https://github.com/PyAutoLabs/HowToLens/blob/main/notebooks/chapter_1_introduction/tutorial_1_grids_and_galaxies.ipynb).


In [ ]:
# @title Set up Google Colab { display-mode: "form" }
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("howtolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

import numpy as np
from pathlib import Path

import autofit as af
import autolens as al
import autolens.plot as aplt

__Shear Catalogues__

A shear catalogue contains one row per background galaxy, including:

- Its observed sky position, expressed here as `(y, x)` offsets in arc-seconds.
- Two components describing its estimated lensing distortion.
- An uncertainty that accounts for intrinsic shape dispersion and measurement noise.

Real catalogues also require information about source redshifts, selection and calibration. The lensing signal depends on the geometry between the observer, lens and source, just as it did for the multiple source redshifts in the cluster tutorial.

Producing these measurements begins with telescope images. Galaxies must be detected and selected, and their shapes measured while accounting for the PSF, noise and instrumental effects. The PSF can both blur galaxy shapes and introduce coherent distortions that could be mistaken for lensing.

We start after this shape-measurement stage. The dataset supplied to this tutorial contains catalogue measurements rather than image pixels, so our likelihood does not convolve a model image with a PSF. This does not make observational systematics disappear: their calibration and uncertainty remain part of a real weak-lensing analysis.

There is also an important distinction between **shear** and **reduced shear**. Galaxy-shape measurements generally estimate the reduced shear:

\[
g = \frac{\gamma}{1-\kappa}.
\]

When the convergence is sufficiently small, reduced shear is approximately equal to shear. This tutorial uses an idealized catalogue of shear measurements generated directly from the model. When working with observed catalogues, check which quantity the measurements estimate and ensure that the model predicts the same quantity.

__Mass Scales__

Weak lensing is useful across the mass scales explored in this chapter.

For individual galaxy lenses, the available background galaxies usually provide too little information for a precise halo measurement. **Galaxy–galaxy lensing** therefore commonly combines measurements around many foreground galaxies to constrain the average mass distribution of a population.

Groups produce larger signals, but the precision obtainable for an individual system depends on its mass, redshift and the depth and quality of the observations. Combining groups can also improve the measurement.

Massive clusters can produce measurable coherent distortions across hundreds or thousands of background galaxies. These measurements extend well beyond the central arcs and help constrain the cluster’s extended dark matter halo.

There is no sharp Einstein-radius boundary separating these applications. Detectability depends on the lensing geometry, the radii sampled, the number of usable sources and their shape uncertainties.

We use a cluster-scale lens here because it provides a useful demonstration of an individual-system weak-lensing analysis.

__Ray Tracing__

We begin with a `Tracer`, using an elliptical `Isothermal` mass profile with an Einstein radius of 25.0 arc-seconds.

This is a deliberately simple cluster-scale mass distribution. Real cluster analyses often use NFW dark matter halos and additional mass components, as discussed in the previous tutorial.

The lens has no light profile because this fit uses catalogue measurements rather than an image of the lens.

The source galaxy also has no light profile. It supplies the source-plane redshift needed to calculate the lensing geometry; it does not represent the detailed appearance of an individual background galaxy.

For simplicity, all background galaxies in this simulation lie at redshift 1.0, behind a lens at redshift 0.5.

In [ ]:
lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        einstein_radius=25.0,
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.8, angle=45.0),
    ),
)

source_galaxy = al.Galaxy(redshift=1.0)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

__Source Galaxy Positions__

We draw 1500 observed background-galaxy positions in an annulus extending from 50 to 200 arc-seconds from the lens centre, distributed uniformly in area.

This excludes the central region containing the strongest lensing features. In real observations, the usable region also depends on blending, cluster-member contamination and the reliability of the shape measurements.

The annulus covers approximately 33 square arc-minutes, giving a source density of about 46 galaxies per square arc-minute. This represents a deep illustrative catalogue; the usable density in an observation depends on its depth, resolution and selection criteria.

For comparison, a spherical isothermal lens with an Einstein radius of 25 arc-seconds has shear and convergence of 0.25 at 50 arc-seconds, falling to about 0.06 at 200 arc-seconds. Our elliptical model varies with position angle as well.

The inner boundary is therefore outside the Einstein radius, but it does not guarantee that reduced shear is approximately equal to shear. For the spherical example, the reduced shear at 50 arc-seconds is about 0.33. Remember that our simulation generates shear directly rather than simulating observed galaxy ellipticities.

The positions are stored in an `al.Grid2DIrregular`, using the same `(y, x)` coordinate convention as the point-source tutorials. These are observed image-plane positions at which we evaluate the lensing field, not reconstructed source-plane coordinates.

In [ ]:
rng = np.random.default_rng(1)

n_galaxies = 1500
radius_inner = 50.0  # arc-seconds — excludes the central region of the simulated lens.
radius_outer = 200.0  # arc-seconds — the edge of our simulated weak-lensing field.

radii = np.sqrt(rng.uniform(radius_inner**2.0, radius_outer**2.0, n_galaxies))
phi = rng.uniform(0.0, 2.0 * np.pi, n_galaxies)

positions = al.Grid2DIrregular(
    values=np.stack([radii * np.sin(phi), radii * np.cos(phi)], axis=1)
)

__Simulate__

`SimulatorShearYX` evaluates the tracer’s shear field at each catalogue position and adds independent Gaussian noise to its two components.

We use `noise_sigma=0.25` per component as an illustrative shape-noise dispersion. The intrinsic shapes of real galaxies produce substantial scatter, and measurement errors add further uncertainty.

This is a simplified statistical simulation. We do not generate images of individual background galaxies or run a shape-measurement pipeline. Instead, we directly generate noisy estimates of the shear field that our likelihood will fit.

The simulator’s default configuration generates shear rather than reduced shear, matching the idealization described above.

We keep the catalogue in memory and pass it directly to the fitting tools. The script `scripts/simulator/weak_lensing.py`, used for the weak-lensing example in Chapter 1, demonstrates saving a catalogue to disk.

In [ ]:
dataset_name = "weak_lensing"

simulator = al.SimulatorShearYX(noise_sigma=0.25, seed=1)

dataset = simulator.via_tracer_from(
    tracer=tracer,
    grid=positions,
    name=dataset_name,
)

print(dataset.info)

__Visualize__

We display the catalogue as a **whisker map**. Each galaxy position has a line segment whose length represents the magnitude of its measured shear estimate and whose orientation represents the corresponding position angle.

The segments have no arrowheads. A shear orientation is unchanged by a rotation of 180 degrees, so a line represents it more appropriately than a directed arrow.

Around a centrally concentrated lens, the coherent pattern is predominantly tangential: the whiskers tend to follow circles around the lens centre. Ellipticity and additional mass structure can modify that pattern.

Shape noise introduces substantial scatter in both the lengths and orientations of the plotted whiskers. Individual measurements need not align with the underlying field, even when the ensemble constrains it well.

The plot provides an initial view of the measurements. Later, we will compare them with a model and summarize their tangential component in radial bins.

In [ ]:
aplt.plot_shear_yx_2d(shear_yx=dataset.shear_yx)

`aplt.subplot_weak_dataset` summarizes the catalogue in four panels: the shear whiskers, the per-galaxy uncertainties, the measured shear magnitudes and their position angles.

Our simulation assigns the same uncertainty to every galaxy. Real catalogues generally have varying uncertainties because galaxies differ in brightness, size and the quality of their shape measurements.

The magnitude and position-angle panels describe the noisy measurements. They should not be interpreted as direct maps of the underlying lensing field.

In [ ]:
aplt.subplot_weak_dataset(dataset=dataset)

__Mass Map__

The shear field contains information about the convergence `kappa`, the dimensionless projected mass density introduced earlier in the series.

The **Kaiser–Squires** method uses the relationship between shear and convergence in Fourier space to reconstruct a convergence map without choosing a parametric mass profile such as an Isothermal or NFW.

`aplt.plot_convergence_map` bins the catalogue onto a regular grid, smooths the binned shear field and performs this reconstruction.

The resulting map provides a useful view of the projected mass structure. It traces the total lensing mass, including both dark matter and baryons; identifying a feature specifically with dark matter requires additional information.

Avoid interpreting every peak as a real mass concentration. Shape noise, smoothing, the finite field and gaps in the catalogue all affect the reconstruction. Our catalogue has no measurements inside its central hole, so the displayed map there is not a direct local measurement.

Shear also leaves an additive constant in the reconstructed convergence unconstrained. This is related to the mass-sheet degeneracy discussed in Chapter 3.

Use this map as an exploratory diagnostic. Quantitative measurements require uncertainty estimates and careful treatment of the survey geometry. Here, we proceed by fitting a parametric mass model directly to the catalogue.

In [ ]:
aplt.plot_convergence_map(
    shear_yx=dataset.shear_yx,
    shape_native=(30, 30),
    smoothing_sigma_pixels=1.0,
)

__Fitting__

We now follow the fitting workflow introduced in Chapter 1: specify a model `Tracer`, combine it with the dataset in a fit object, and inspect the residuals and log likelihood.

For this dataset, the fit object is `FitWeak`. It evaluates the predicted shear at each observed galaxy position and compares its two components with the catalogue measurements.

Our likelihood assumes independent Gaussian errors for the two components, using the supplied uncertainties. This matches the way we generated this simulated catalogue.

We first supply the input tracer used by the simulator. Its predictions describe the underlying shear field, while the residuals contain the Gaussian noise added during simulation.

There are two measured components per galaxy, giving 3000 scalar measurements. At this fixed input model, the expected chi-squared is therefore 3000, with statistical fluctuations between noise realizations.

This count should not be confused with the residual degrees of freedom after estimating parameters from the same data. We will shortly allow the mass parameters to vary in a non-linear search.

Unlike an imaging likelihood, this calculation does not generate and convolve a model image or reconstruct source surface brightness. Its main task is evaluating the lensing field at the catalogue positions and calculating the likelihood.

The computational cost depends on the catalogue size, mass model and implementation, but this makes catalogue fitting a useful complement to the imaging and point-source analyses explored earlier.

In [ ]:
fit = al.FitWeak(dataset=dataset, tracer=tracer)

print()
print("Fit Summary (true model)")
print("------------------------")
print(f"n_galaxies        : {dataset.n_galaxies}")
print(f"n_data_components: {2 * dataset.n_galaxies}")
print(f"chi_squared       : {fit.chi_squared:.3f}")
print(f"log_likelihood    : {fit.log_likelihood:.3f}")

We next halve the model’s Einstein radius while keeping its centre and ellipticity fixed.

For this Isothermal profile, that change halves the predicted shear at every catalogue position. The model therefore underpredicts the coherent signal present in the simulated data.

Compare the two printed log likelihoods. This deliberately large change illustrates how the catalogue distinguishes different mass normalizations, even though individual galaxy measurements are noisy.

The likelihood combines information from the whole catalogue: the preference does not require every individual measurement to lie closer to the input model.

In [ ]:
tracer_wrong = al.Tracer(
    galaxies=[
        al.Galaxy(
            redshift=0.5,
            mass=al.mp.Isothermal(
                centre=(0.0, 0.0),
                einstein_radius=12.5,
                ell_comps=al.convert.ell_comps_from(axis_ratio=0.8, angle=45.0),
            ),
        ),
        source_galaxy,
    ]
)

fit_wrong = al.FitWeak(dataset=dataset, tracer=tracer_wrong)

print()
print(f"log_likelihood (true model) : {fit.log_likelihood:.3f}")
print(f"log_likelihood (wrong model): {fit_wrong.log_likelihood:.3f}")

`aplt.subplot_fit_weak` displays the observed shear field, the model field, an overlay of the two and the per-galaxy chi-squared contributions.

The input model produces a smooth, coherent pattern. The observed whiskers scatter around that pattern because of shape noise, so a good fit does not require close agreement for every galaxy.

Likewise, the chi-squared map need not look uniform. Individual galaxies can have large contributions simply because their noise realizations are large.

Look for coherent discrepancies across the field, alongside the overall likelihood and residual statistics. Such patterns can indicate missing mass structure, an inadequate model or problems with the measurements.

For this simulation, the input model and noise assumptions match the data-generation process, providing a reference for interpreting these diagnostics.

In [ ]:
aplt.subplot_fit_weak(fit=fit)

__Model Fit__

We now infer the mass-model parameters using the workflow introduced in Chapter 2: compose a model with `af.Model` and `af.Collection`, choose a non-linear search, create an analysis object and fit.

`AnalysisWeak` builds a `Tracer` from each sampled parameter set and evaluates the catalogue likelihood using the same `FitWeak` calculation we inspected above.

Our Isothermal model has five non-linear parameters:

- Two centre coordinates.
- Two ellipticity components.
- The Einstein radius.

The lens and source redshifts remain fixed. Neither galaxy has a light profile, so there are no light-profile parameters or source reconstructions to fit.

Earlier in this chapter, we often fixed individual galaxy centres to their measured centres of light. Here, the mass profile represents the cluster as a whole, and we allow its centre to vary. A cluster’s overall mass centre need not coincide exactly with the light centre of its brightest galaxy.

We assign an Einstein-radius prior spanning 0–60 arc-seconds and Gaussian centre priors with a width of 20 arc-seconds around the origin. These are illustrative choices suited to this simulated system.

As in the group and cluster tutorials, check that the priors describe the scale and plausible configurations of the system being analyzed. Inspect the resulting posterior to determine whether the data constrain these parameters within their allowed ranges.

In [ ]:
mass = af.Model(al.mp.Isothermal)
mass.einstein_radius = af.UniformPrior(lower_limit=0.0, upper_limit=60.0)
mass.centre.centre_0 = af.GaussianPrior(mean=0.0, sigma=20.0)
mass.centre.centre_1 = af.GaussianPrior(mean=0.0, sigma=20.0)

lens = af.Model(al.Galaxy, redshift=0.5, mass=mass)

source = af.Model(al.Galaxy, redshift=1.0)

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

print(model.info)

The model figure summarizes this structure: one lens mass profile with five free parameters and a source galaxy supplying a fixed redshift.

Use it alongside `model.info`, which lists the priors and fixed values.

The single source galaxy in this model does not mean that the catalogue contains only one background galaxy. It supplies the common source-plane redshift assumed for all 1500 catalogue entries.

Their individual observed positions and shear estimates belong to the dataset; we do not introduce a separate light profile or free source position for each entry.

In [ ]:
af.ModelPlotter(model).figure()

We use Nautilus to sample the posterior, as in the earlier tutorials, and pass the catalogue to `AnalysisWeak`.

Each likelihood evaluation predicts the shear field at the 1500 galaxy positions and compares it with the measurements. There is no image convolution or source reconstruction in this calculation.

This is a compact five-parameter example, but its runtime still depends on the implementation, hardware and number of likelihood evaluations required by the search.

Use the search outputs to monitor progress and assess convergence before interpreting the inferred parameters. The familiar output directory contains the model information, samples and fit diagnostics.

In [ ]:
search = af.Nautilus(
    path_prefix=Path("howtolens", "chapter_4"),
    name="tutorial_6_weak_lensing",
    unique_tag=dataset_name,
    n_live=100,
    iterations_per_quick_update=5000,
)

analysis = al.AnalysisWeak(dataset=dataset)

print(
    "The non-linear search has begun. Check "
    "output/howtolens/chapter_4/tutorial_6_weak_lensing "
    "for live results, images and model information. "
    "Continue once the search has completed."
)

result = search.fit(model=model, analysis=analysis)

print("The search has finished. You may now continue the notebook.")

__Result__

Inspect `result.info` to see the inferred parameters and their uncertainties.

Because we simulated this catalogue, we can compare them with the input Einstein radius of 25 arc-seconds, axis ratio of 0.8, position angle of 45 degrees and centre at the origin.

The code parameterizes ellipticity using two components, so convert between those components and axis ratio and position angle when making that comparison.

Noise means that the inferred values need not equal the inputs exactly. Consider the posterior intervals and parameter correlations, rather than judging recovery from a single fitted realization.

The fit subplot below displays the maximum-likelihood realization found by the search. Compare its coherent shear pattern with the noisy measurements, using the diagnostics discussed above.

These constraints are conditional on our chosen mass profile, fixed redshifts and noise assumptions. A real cluster may require a more flexible mass distribution and additional treatment of observational uncertainties.

In [ ]:
print(result.info)

aplt.subplot_fit_weak(fit=result.max_log_likelihood_fit)

Weak-lensing measurements around clusters are often summarized using a **tangential shear profile**.

For each background galaxy, we resolve its shear into components defined relative to a chosen centre. The tangential component describes the tendency for images to stretch around that centre.

`aplt.plot_shear_profile` groups the catalogue into radial bins and plots the mean tangential component with uncertainties. Passing a fit also overlays the model’s binned tangential profile.

Here we use eight bins around the simulation’s input centre, `(0.0, 0.0)`. This plotting choice does not fix the mass centre in the fit: the model still has two free centre coordinates.

The plot is a summary of the fitted catalogue. Our likelihood used both shear components at every galaxy position, retaining angular information that radial averaging can hide.

The second set of points shows the **cross component**, defined at 45 degrees to the tangential component. Its azimuthal average is expected to be consistent with zero for the ideal lensing signal and suitable angular sampling.

A significant cross signal therefore motivates checks of the measurements, angular coverage and model geometry. However, a cross profile consistent with zero does not establish that all systematic errors are absent; some biases can change the tangential signal without producing a cross signal.

Use the radial profiles together with the two-dimensional fit diagnostics and posterior uncertainties.

In [ ]:
aplt.plot_shear_profile(
    result.max_log_likelihood_fit,
    centre=(0.0, 0.0),
    bins=8,
)

__Joint Strong and Weak Lensing__

We fitted the weak-lensing catalogue on its own. When a system also has strong-lensing observations, combining the two can constrain its mass distribution across a wider range of radii.

**Strong lensing** provides detailed information where multiple images and arcs form. It often tightly constrains the enclosed projected mass near the Einstein radius, while the inferred density profile and division between mass components remain subject to degeneracies.

**Weak lensing** adds measurements over a larger field, extending our sensitivity to the outer dark matter halo. Its constraining power depends on the background-source density, redshifts, shape uncertainties and area observed.

These are complementary measurements of the same gravitational field. Strong lensing does not become insensitive to all mass outside the Einstein radius, and weak lensing does not independently determine every feature of the outer halo.

A joint analysis uses shared mass-model parameters to predict both datasets. We combine an imaging or point-source analysis with `AnalysisWeak`, following the multi-dataset approach used earlier in this chapter.

For datasets whose measurement errors can be treated as independent, their log likelihoods add. Each analysis retains its own data model and uncertainties, while the shared mass parameters must explain both sets of observations.

The source populations need not have the same redshift. Each prediction must use the geometry appropriate to its sources, just as in the previous cluster tutorial.

For example, strong-lensing images can constrain the central mass distribution while a weak-lensing catalogue helps constrain the mass and concentration of an extended NFW halo. Combining them can reduce degeneracies, although the result still depends on the chosen mass components and observational assumptions.

Inspect the contributions from both datasets separately. A satisfactory joint fit should describe both, and a disagreement can reveal missing mass structure, calibration problems or overly restrictive assumptions.

The `autolens_workspace/scripts/weak` directory provides examples for continuing this work:

- [start_here.py](https://github.com/PyAutoLabs/autolens_workspace/blob/main/scripts/weak/start_here.py): an introduction using a real galaxy-shape catalogue for Abell 2744.
- [simulator.py](https://github.com/PyAutoLabs/autolens_workspace/blob/main/scripts/weak/simulator.py), [fit.py](https://github.com/PyAutoLabs/autolens_workspace/blob/main/scripts/weak/fit.py) and [modeling.py](https://github.com/PyAutoLabs/autolens_workspace/blob/main/scripts/weak/modeling.py): simulation, fitting and inference examples.
- [likelihood_function.py](https://github.com/PyAutoLabs/autolens_workspace/blob/main/scripts/weak/likelihood_function.py): a step-by-step explanation of the weak-lensing likelihood.
- [features/strong_lensing](https://github.com/PyAutoLabs/autolens_workspace/tree/main/scripts/weak/features/strong_lensing): examples combining strong- and weak-lensing measurements.

__Wrap Up__

This tutorial completes Chapter 4 and the four core chapters of **HowToLens**.

We have learned how to:

- Represent weak-lensing measurements as a catalogue of positions, shear estimates and uncertainties.
- Simulate an idealized catalogue and understand how shape noise affects its measurements.
- Visualize the shear field and construct an exploratory convergence map.
- Evaluate a catalogue likelihood and infer mass-model parameters using the familiar search workflow.
- Interpret fit diagnostics and posterior uncertainties, and understand the motivation for joint strong- and weak-lensing analyses.

The distinction between shear and reduced shear matters when moving from this idealized simulation to observed galaxy shapes. So do source redshifts, calibration, selection and the flexibility of the mass model.

Across the series, the same ideas have supported increasingly complex analyses.

**Chapter 1** introduced grids, light and mass profiles, ray tracing and fitting data. These calculations came together as a likelihood function.

**Chapter 2** used that likelihood for inference: defining priors, running non-linear searches, examining posterior uncertainties and comparing models.

**Chapter 3** introduced pixelized source reconstructions, allowing more flexible descriptions of source morphology while controlling that flexibility through regularization.

**Chapter 4** extended the modeling to additional galaxies, shared scaling relations, groups, clusters and weak lensing. We saw how fixed measurements and shared relations can make large models tractable, while also shaping the conclusions we can draw from them.

You now have a foundation for building lens models, understanding their likelihoods and assessing their results. Applying those tools to a scientific question means choosing an appropriate model, checking the data and testing which conclusions remain robust when assumptions change.

The next step is the [autolens_workspace](https://github.com/PyAutoLabs/autolens_workspace). Start with the example closest to your dataset, then use its feature examples and likelihood guides to build up the analysis you need.

The workspace covers imaging, interferometry, point sources, groups, clusters and weak lensing, alongside tools such as the automated SLaM pipelines.

Bring your own data, keep inspecting what the model is doing, and enjoy exploring the Universe through gravitational lensing!